# invoke的调用
简单来说， invoke 方法的作用就是：
1. 接收你的输入（问题、指令、对话历史等）
2. 发送给 LLM 模型（如 GPT-4、Llama、Claude 等）
3. 返回模型的响应（文本回复 + 元数据信息）

基本语法：

response = model.invoke(input, config=None)

## 1. 字典列表传递参数调用
举例1：采用字典列表传递参数调用（至于文档的调用，前面的例子已经有了）

In [1]:
from dotenv import load_dotenv
import os

from langchain.chat_models import init_chat_model

load_dotenv(override=True)

API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

#初始化
model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=API_KEY,
    base_url=BASE_URL,
)

msg = [
    {'role': 'system', 'content': '你是一个专业的数学老师'},
    {'role': 'user', 'content': '什么是斐波那契数列？'}
]

res = model_openai.invoke(msg)

print(f'AI的回复：{res.content}')


AI的回复：你好！作为数学老师，我很高兴为你系统梳理**斐波那契数列（Fibonacci Sequence）**。它不仅是初等数学中的经典递推例子，更是贯穿数论、组合数学、自然科学与计算机科学的桥梁。我们按“定义→公式→历史→性质→应用→学习建议”五步展开：

---
### 🔢 1. 基本定义
斐波那契数列是一组**从第三项起，每一项都等于前两项之和**的数字序列。最标准的起始方式为：
```
0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, …
```
> 📌 注：部分中学教材从 `1, 1` 开始（即 `1, 1, 2, 3, 5…`），本质是索引偏移一位，数学性质完全一致。

---
### 📐 2. 数学表达
- **递推关系**：  
  `F(n) = F(n-1) + F(n-2)`  （`n ≥ 2`）
- **初始条件**（现代数学常用约定）：  
  `F(0) = 0,  F(1) = 1`
- **闭式通项（比内公式 / Binet's Formula）**：  
  ```math
  F(n) = \frac{\varphi^n - \psi^n}{\sqrt{5}}
  ```
  其中 `\varphi = \frac{1+\sqrt{5}}{2} ≈ 1.6180339…`（黄金比例），`\psi = \frac{1-\sqrt{5}}{2} ≈ -0.6180339…`。  
  由于 `|ψ| < 1`，当 `n` 增大时 `ψⁿ → 0`，故 `F(n)` 极其接近 `\frac{\varphi^n}{\sqrt{5}}`。这也是为什么数列增长看似“平滑”，实则由无理数驱动。

---
### 📜 3. 历史起源
该数列得名于13世纪意大利商人兼数学家**莱昂纳多·斐波那契**（Leonardo Fibonacci）。1202年他在《算盘书》（*Liber Abaci*）中提出一个**理想化兔子繁殖问题**：
> 假设刚出生的兔子需一个月成熟，此后每月固定生一对新兔，且兔子永不死亡。问：第 `n` 个月共有多少对兔子？

解出的月兔子对数恰为斐波那契数列。虽不符合真实生物学（忽略死亡率、生育周期等），但该问题首次将“累加递推”模式引入欧洲数学体系，后世遂以其姓氏命名。

---
##

举例2：多轮对话带历史

In [2]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=API_KEY,
    base_url=BASE_URL,
)

msg = [
    {"role":"system","content":"你是一个专业的数学老师。"},
    {"role":"user","content":"2 + 3 * 2 = ？"},
    {"role":"assistant","content":"8"},
    {"role":"user","content":"我刚才问了什么问题？"}
]

res = model_openai.invoke(msg)

print(f'AI的专业回答：{res.content}')


AI的专业回答：你刚才问的是：**“2 + 3 * 2 = ？”**


举例3：传递对话记忆

In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=API_KEY,
    base_url=BASE_URL,
)

conversation = [
    {'role': 'system', 'content': '你是一个非常友好的AI助手'},
    {'role': 'user', 'content': '你好，我叫深秋'}
]

# 第一次对话
res1 = model_openai.invoke(conversation)
print(res1.content)

#添加记忆,对这个conversation列表进行追加元素
conversation.append({'role': 'assistant', 'content': res1.content})

conversation.append({'role': 'user', 'content': '我叫什么名字？'})
#第二次对话调用
res2 = model_openai.invoke(conversation)
print(res2.content)

你好，深秋！很高兴认识你～🍂  
“深秋”这个名字听起来特别有诗意，像一幅宁静又温暖的画面。今天有什么我可以帮你的吗？无论是解答问题、找灵感，还是随便聊聊天，我都在这儿陪你哦！😊
你叫**深秋**呀！🍂 这是我开场聊天时你告诉我的～  
今天想聊点什么，或者有什么我能帮你的吗？随时告诉我哦！😊


## 2. 消息对象列表
使用内置的消息类（如SystemMessage, HumanMessage, AIMessage）,将消息对象列表输入模型。

适用场景：需要类型检查（针对大型项目）、IDE自动不全的场景

缺点：代码较长，不如字典简洁，难以序列化（JSON）
| 消息类 | 对应字典格式 | 作用 |
| ---- | ---- | ---- |
| SystemMessage | {'role': 'system', ...} | 系统提示 |
| HumanMessage | {'role': 'user', ...} | 用户输入 |
| AIMessage | {'role': 'assistant', ...} | AI回复 |

举例1：

In [5]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

#先加载配置文件
load_dotenv(override=True)

API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model_openai = init_chat_model(
    model='qwen3.7-flash',
    model_provider='openai',
    api_key=API_KEY,
    base_url=BASE_URL,
)

msg = [
    SystemMessage('你是一个专业的数学老师。'),
    HumanMessage('2 + 3*2 = ?'),
]

res = model_openai.invoke(msg)

#继续对话
msg.append(AIMessage(content=res.content))
msg.append(HumanMessage(content='能给出一个例子吗？'))

print(res.content)


正确答案是 **8**。

**解析：**
在数学运算中，需要遵循**“先乘除，后加减”**的运算顺序。
- 第一步：先算乘法 `3 × 2 = 6`
- 第二步：再算加法 `2 + 6 = 8`

所以，`2 + 3 × 2 = 8`。如有其他题目或疑问，欢迎继续提问！
